# Digital Communication Laboratory
## Experiment 11: M-ary Phase Shift Keying (M-PSK) and M-ary Quadrature Amplitude Modulation (M-QAM)

**Programme:** B.Tech in Electronics and Communication Engineering  
**Platform:** Python 3 (NumPy, SciPy, Matplotlib, Pandas)  
**Execution Environment:** Google Colab & Local Jupyter Notebook  
**Mode:** Self-Contained First-Principles Simulation without Proprietary Toolboxes  

---

### Objectives & Syllabus Coverage:
1. **Constellation Synthesis & Normalization:** Programmatically construct QPSK, 8-PSK, 16-QAM, and 64-QAM constellations and normalize their average symbol energies strictly to $E_s = 1.0$.
2. **Mandatory Syllabus Validation:** Compute and report numerical average constellation energy before and after normalization across all four modulation orders.
3. **Minimum Euclidean Distance Detection:** Implement an optimal maximum-likelihood vector slicer based on Euclidean distance metric minimization.
4. **Received Cloud Analysis:** Examine received 2D symbol clouds under AWGN channel conditions across SNR levels ($0\text{ dB}, 5\text{ dB}, 10\text{ dB}, 20\text{ dB}$).
5. **Decision Regions & Voronoi Geometry:** Visualize 2D decision boundaries: radial pie wedges for 8-PSK vs square Voronoi cells for 16-QAM.
6. **Gray vs Natural Mapping Comparison:** Quantify the Bit Error Rate penalty when using natural (binary) mapping versus Gray coding on 16-QAM.
7. **Peak-to-Average Power Ratio (PAPR):** Characterize constellation power distribution and the concentric energy tiers of 16-QAM.
8. **Spectral Efficiency vs Power Trade-Off:** Plot required $E_b/N_0$ at $\text{BER} = 10^{-3}$ against bandwidth efficiency $\eta = \log_2(M)$ and benchmark against the Shannon capacity limit.
9. **Theoretical vs Monte Carlo SER/BER Performance:** Validate simulated SER and BER against analytical error formulas across $E_b/N_0 \in [0, 16]\text{ dB}$.

In [ ]:
# Cell 1: Environment Setup, Library Imports, and Publication Styling
%matplotlib inline
import os
import sys
from pathlib import Path
from typing import Tuple, Dict, List, Optional

import numpy as np
import scipy.special as sp
import matplotlib.pyplot as plt
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    display = print

# Set plot styling for publication-quality visuals
plt.rcParams.update({
    'font.family': 'serif',
    'font.size': 11,
    'axes.labelsize': 12,
    'axes.titlesize': 13,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
    'figure.titlesize': 14,
    'lines.linewidth': 1.5,
    'grid.alpha': 0.5,
    'grid.linestyle': '--',
    'savefig.dpi': 300,
    'savefig.bbox': 'tight'
})

OUTPUT_DIR = Path('outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('[OK] Environment initialized successfully.')
print(f'     Output directory locked at: {OUTPUT_DIR.resolve()}')
print('     All 8 figures will be displayed inline and saved to outputs/.')

---
## 1. Constellation Synthesis, Normalization & ML Detection (First Principles)

- **M-PSK:** All points lie on the unit circle: $s_m = \exp(j(2\pi m / M + \pi/M))$. Unnormalized energy is already $E_{\text{avg}} = 1.0$.
- **Square M-QAM:** Grid coordinates in $\{\pm 1, \pm 3, \dots, \pm (\sqrt{M} - 1)\}$.
  Unnormalized average energy: $E_{\text{unnorm}} = \frac{2}{3}(M - 1)$.
  - For 16-QAM: $E_{\text{unnorm}} = \frac{2}{3}(15) = 10.0 \implies c = 1/\sqrt{10} \approx 0.3162$.
  - For 64-QAM: $E_{\text{unnorm}} = \frac{2}{3}(63) = 42.0 \implies c = 1/\sqrt{42} \approx 0.1543$.
- **2D Gray Coding:** Achieved by decomposing square QAM into independent 1D In-Phase and Quadrature Gray codes: $\text{Gray}(n) = n \oplus (n \gg 1)$.
- **Minimum Euclidean Distance Detector:** Maximum-likelihood rule $\hat{m} = \arg\min_m |r - s_m|^2$.

In [ ]:
# Cell 2: First-Principles Constellation Builders, Gray Code Tables, and ML Slicer
def binary_to_gray(n: int) -> int:
    return n ^ (n >> 1)

def generate_gray_code_table(num_bits: int) -> List[str]:
    n_states = 1 << num_bits
    return [format(binary_to_gray(i), f'0{num_bits}b') for i in range(n_states)]

def get_mpsk_constellation(m_order: int) -> Tuple[np.ndarray, np.ndarray, Dict[str, complex], float, float]:
    k = int(np.log2(m_order))
    gray_codes = generate_gray_code_table(k)
    angles = np.array([(2.0 * m + 1.0) * np.pi / m_order for m in range(m_order)])
    unnorm_syms = np.exp(1j * angles)
    unnorm_energy = float(np.mean(np.abs(unnorm_syms) ** 2))
    norm_factor = 1.0 / np.sqrt(unnorm_energy)
    norm_syms = unnorm_syms * norm_factor
    norm_energy = float(np.mean(np.abs(norm_syms) ** 2))
    mapping = {gray_codes[i]: norm_syms[i] for i in range(m_order)}
    return unnorm_syms, norm_syms, mapping, unnorm_energy, norm_energy

def get_mqam_constellation(m_order: int, use_gray: bool = True) -> Tuple[np.ndarray, np.ndarray, Dict[str, complex], float, float]:
    sqrt_m = int(np.sqrt(m_order))
    k = int(np.log2(m_order))
    k_per_axis = k // 2
    levels_1d = 2 * np.arange(sqrt_m) - (sqrt_m - 1)
    code_1d = generate_gray_code_table(k_per_axis) if use_gray else [format(i, f'0{k_per_axis}b') for i in range(sqrt_m)]

    unnorm_list = []
    for i_val in levels_1d:
        for q_val in levels_1d:
            unnorm_list.append(complex(i_val, q_val))
    unnorm_syms = np.array(unnorm_list)
    unnorm_energy = float(np.mean(np.abs(unnorm_syms) ** 2))
    norm_factor = 1.0 / np.sqrt(unnorm_energy)
    norm_syms = unnorm_syms * norm_factor
    norm_energy = float(np.mean(np.abs(norm_syms) ** 2))

    mapping = {}
    idx = 0
    for i in range(sqrt_m):
        for q in range(sqrt_m):
            mapping[code_1d[i] + code_1d[q]] = norm_syms[idx]
            idx += 1
    return unnorm_syms, norm_syms, mapping, unnorm_energy, norm_energy

def demodulate_minimum_distance(rx_symbols: np.ndarray, mapping: Dict[str, complex]) -> Tuple[np.ndarray, np.ndarray]:
    bit_strings = list(mapping.keys())
    constellation = np.array(list(mapping.values()))
    k = len(bit_strings[0])
    rx_col = rx_symbols[:, np.newaxis]
    const_row = constellation[np.newaxis, :]
    dist_sq = np.abs(rx_col - const_row) ** 2
    best_indices = np.argmin(dist_sq, axis=1)
    detected_symbols = constellation[best_indices]
    detected_bits = np.empty(len(rx_symbols) * k, dtype=int)
    for i, idx in enumerate(best_indices):
        b_str = bit_strings[idx]
        for b_pos in range(k):
            detected_bits[i * k + b_pos] = int(b_str[b_pos])
    return detected_bits, detected_symbols

def add_awgn_complex(symbols: np.ndarray, snr_db: float, bits_per_symbol: int, seed: Optional[int] = None) -> np.ndarray:
    eb = 1.0 / bits_per_symbol
    snr_lin = 10.0 ** (snr_db / 10.0)
    n0 = eb / snr_lin
    sigma = np.sqrt(n0 / 2.0)
    rng = np.random.default_rng(seed)
    noise = rng.normal(0.0, sigma, size=len(symbols)) + 1j * rng.normal(0.0, sigma, size=len(symbols))
    return symbols + noise

print('[OK] Constellation builders, ML slicer, and AWGN channel compiled successfully.')

---
## 2. Mandatory Syllabus Validation: Constellation Energy Normalization [Figure 8]

> **Syllabus Requirement:** *"Report numerical average constellation energy before and after normalization."*

We compute the unnormalized average constellation energy $E_{\text{unnorm}} = \frac{1}{M}\sum |s_m|^2$, scale factor $c = 1/\sqrt{E_{\text{unnorm}}}$, and normalized energy $E_s = \frac{1}{M}\sum |c \cdot s_m|^2$ to strictly verify $E_s = 1.000000$ across QPSK, 8-PSK, 16-QAM, and 64-QAM.

In [ ]:
# Cell 3: Mandatory Syllabus Validation - Constellation Energy Normalization [Figure 8]
def validate_constellation_energies() -> pd.DataFrame:
    configs = [
        ('QPSK (4-PSK)', 4, 'PSK', 2),
        ('8-PSK', 8, 'PSK', 3),
        ('16-QAM', 16, 'QAM', 4),
        ('64-QAM', 64, 'QAM', 6)
    ]
    records = []
    for name, order, mod_type, bits in configs:
        if mod_type == 'PSK':
            unnorm_s, norm_s, _, e_unnorm, e_norm = get_mpsk_constellation(order)
            th_unnorm = 1.0
        else:
            unnorm_s, norm_s, _, e_unnorm, e_norm = get_mqam_constellation(order, use_gray=True)
            th_unnorm = 2.0 * (order - 1.0) / 3.0
        scale_factor = 1.0 / np.sqrt(e_unnorm)
        records.append({
            'Modulation': name,
            'Order M': order,
            'Bits/Sym (k)': bits,
            'Theoretical Unnorm E': th_unnorm,
            'Numerical Unnorm E': e_unnorm,
            'Scale Factor': scale_factor,
            'Normalized E (Es)': e_norm,
            'Status': 'PASSED' if np.isclose(e_norm, 1.0, atol=1e-8) else 'FAILED'
        })
    return pd.DataFrame(records)

df_energy = validate_constellation_energies()
print('=== MANDATORY SYLLABUS VALIDATION SUMMARY TABLE ===')
display(df_energy)

# Generate Figure 8: Bar Chart & Verification Table
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))
x = np.arange(len(df_energy))
width = 0.35

ax1.bar(x - width/2, df_energy['Numerical Unnorm E'], width, label='Unnormalized Average Energy', color='crimson')
ax1.bar(x + width/2, df_energy['Normalized E (Es)'], width, label='Normalized Average Energy ($E_s = 1.0$)', color='forestgreen')
ax1.set_xticks(x)
ax1.set_xticklabels(df_energy['Modulation'], fontsize=10)
ax1.set_ylabel('Average Symbol Energy')
ax1.set_title('Mandatory Validation: Constellation Energy Normalization', fontsize=12)
ax1.grid(True, axis='y', linestyle='--', alpha=0.6)
ax1.legend(loc='upper left')

ax2.axis('off')
table_data = []
for _, r in df_energy.iterrows():
    table_data.append([
        r['Modulation'], f"{r['Bits/Sym (k)']}", f"{r['Theoretical Unnorm E']:.2f}",
        f"{r['Numerical Unnorm E']:.2f}", f"{r['Scale Factor']:.4f}",
        f"{r['Normalized E (Es)']:.6f}", r['Status']
    ])
col_labels = ['Modulation', 'k', 'Theory Unnorm', 'Num Unnorm', 'Scale', 'Es (Norm)', 'Status']
table = ax2.table(cellText=table_data, colLabels=col_labels, loc='center', cellLoc='center')
table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1.1, 1.8)
ax2.set_title('Numerical Energy Verification Summary', fontsize=12, pad=15)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / '8_mandatory_energy_normalization_validation.png')
plt.show()
print(f'[OK] Figure 8 saved to: {OUTPUT_DIR / "8_mandatory_energy_normalization_validation.png"}')

---
## 3. Ideal Constellation Geometry & 2D Gray Mapping [Figure 1]

We visualize the ideal normalized constellations ($E_s = 1.0$) for QPSK, 8-PSK, 16-QAM, and 64-QAM.
- Gray code bit sequences are mapped directly to each symbol point.
- The unit circle $|s| = 1.0$ illustrates constant envelope transmission in M-PSK versus multi-amplitude grid packing in M-QAM.

In [ ]:
# Cell 4: Ideal Constellations for All Modulation Orders [Figure 1]
fig, axes = plt.subplots(2, 2, figsize=(13, 13))

# (a) QPSK
_, qpsk_s, qpsk_map, _, _ = get_mpsk_constellation(4)
axes[0, 0].scatter(np.real(qpsk_s), np.imag(qpsk_s), color='royalblue', s=80, edgecolors='black', linewidth=1.5)
for b_str, sym in qpsk_map.items():
    axes[0, 0].annotate(f"'{b_str}'", (np.real(sym), np.imag(sym)), textcoords='offset points',
                        xytext=(10, 10), fontsize=10, fontweight='bold', color='darkblue')
axes[0, 0].set_title(r'QPSK (4-PSK) Constellation ($E_s = 1.0, \eta = 2$ b/s/Hz)')

# (b) 8-PSK
_, psk8_s, psk8_map, _, _ = get_mpsk_constellation(8)
axes[0, 1].scatter(np.real(psk8_s), np.imag(psk8_s), color='forestgreen', s=80, edgecolors='black', linewidth=1.5)
for b_str, sym in psk8_map.items():
    axes[0, 1].annotate(f"'{b_str}'", (np.real(sym), np.imag(sym)), textcoords='offset points',
                        xytext=(8, 8), fontsize=9, fontweight='bold', color='darkgreen')
axes[0, 1].set_title(r'8-PSK Constellation ($E_s = 1.0, \eta = 3$ b/s/Hz)')

# (c) 16-QAM
_, qam16_s, qam16_map, _, _ = get_mqam_constellation(16, use_gray=True)
axes[1, 0].scatter(np.real(qam16_s), np.imag(qam16_s), color='crimson', s=70, edgecolors='black', linewidth=1.2)
for b_str, sym in qam16_map.items():
    axes[1, 0].annotate(f"'{b_str}'", (np.real(sym), np.imag(sym)), textcoords='offset points',
                        xytext=(5, 5), fontsize=8, fontweight='bold', color='maroon')
axes[1, 0].set_title(r'16-QAM Gray Constellation ($E_s = 1.0, \eta = 4$ b/s/Hz)')

# (d) 64-QAM
_, qam64_s, _, _, _ = get_mqam_constellation(64, use_gray=True)
axes[1, 1].scatter(np.real(qam64_s), np.imag(qam64_s), color='purple', s=30, alpha=0.9, edgecolors='black', linewidth=0.8)
axes[1, 1].set_title(r'64-QAM Gray Constellation ($E_s = 1.0, \eta = 6$ b/s/Hz)')

for row in axes:
    for ax in row:
        ax.axhline(0, color='black', linestyle='--', alpha=0.5)
        ax.axvline(0, color='black', linestyle='--', alpha=0.5)
        circle = plt.Circle((0, 0), 1.0, color='gray', fill=False, linestyle=':', alpha=0.7)
        ax.add_artist(circle)
        ax.set_xlim(-1.8, 1.8)
        ax.set_ylim(-1.8, 1.8)
        ax.set_xlabel('In-Phase (I)')
        ax.set_ylabel('Quadrature (Q)')
        ax.grid(True)
        ax.set_aspect('equal')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / '1_ideal_constellations_all_orders.png')
plt.show()
print(f'[OK] Figure 1 saved to: {OUTPUT_DIR / "1_ideal_constellations_all_orders.png"}')

---
## 4. Received Constellation Clouds across SNR Regimes [Figure 2]

Under additive white Gaussian noise, transmitted symbols expand into 2D circular clusters. We observe the dispersion across $E_b/N_0 \in \{0\text{ dB}, 5\text{ dB}, 10\text{ dB}, 20\text{ dB}\}$ comparing 16-QAM and 8-PSK. At $0\text{ dB}$, clouds heavily overlap across decision boundaries, producing errors, while at $20\text{ dB}$, symbols form tight, clearly separated clusters.

In [ ]:
# Cell 5: Received Symbol Clouds under AWGN across SNR Sweep [Figure 2]
_, qam16_s, _, _, _ = get_mqam_constellation(16)
_, psk8_s, _, _, _ = get_mpsk_constellation(8)

snr_levels = [0.0, 5.0, 10.0, 20.0]
n_pts = 2500
rng = np.random.default_rng(101)
qam16_stream = qam16_s[rng.integers(0, 16, size=n_pts)]
psk8_stream = psk8_s[rng.integers(0, 8, size=n_pts)]

fig, axes = plt.subplots(2, 4, figsize=(16, 8.5))

for i, snr in enumerate(snr_levels):
    # 16-QAM
    rx_qam = add_awgn_complex(qam16_stream, snr, bits_per_symbol=4, seed=200 + i)
    axes[0, i].scatter(np.real(rx_qam), np.imag(rx_qam), color='crimson', alpha=0.25, s=6)
    axes[0, i].scatter(np.real(qam16_s), np.imag(qam16_s), color='black', marker='+', s=60, linewidth=2)
    axes[0, i].set_title(f'16-QAM Cloud ($E_b/N_0 = {snr:.0f}$ dB)')

    # 8-PSK
    rx_psk = add_awgn_complex(psk8_stream, snr, bits_per_symbol=3, seed=300 + i)
    axes[1, i].scatter(np.real(rx_psk), np.imag(rx_psk), color='forestgreen', alpha=0.25, s=6)
    axes[1, i].scatter(np.real(psk8_s), np.imag(psk8_s), color='black', marker='+', s=60, linewidth=2)
    axes[1, i].set_title(f'8-PSK Cloud ($E_b/N_0 = {snr:.0f}$ dB)')

for row in axes:
    for ax in row:
        ax.axhline(0, color='black', linestyle='--', alpha=0.5)
        ax.axvline(0, color='black', linestyle='--', alpha=0.5)
        ax.set_xlim(-1.8, 1.8)
        ax.set_ylim(-1.8, 1.8)
        ax.set_xlabel('In-Phase (I)')
        ax.set_ylabel('Quadrature (Q)')
        ax.grid(True)
        ax.set_aspect('equal')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / '2_received_constellation_clouds_snr_sweep.png')
plt.show()
print(f'[OK] Figure 2 saved to: {OUTPUT_DIR / "2_received_constellation_clouds_snr_sweep.png"}')

---
## 5. Minimum Euclidean Distance Decision Regions & Voronoi Boundaries [Figure 3]

The maximum-likelihood detector partitions the 2D complex plane into $M$ decision regions:
- **8-PSK:** Forms 8 radial pie wedges centered at each constellation phase, bounded by angular bisectors.
- **16-QAM:** Forms 16 square Voronoi cells. Crucially, because the cells are Cartesian rectangles, 2D detection can be separated into two independent 1D threshold slicings on the In-Phase and Quadrature rails, dramatically reducing computational complexity from $O(M)$ to $O(2\sqrt{M})$.

In [ ]:
# Cell 6: 2D Minimum Euclidean Distance Decision Regions [Figure 3]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6.5))

grid_res = 200
x_lin = np.linspace(-1.8, 1.8, grid_res)
y_lin = np.linspace(-1.8, 1.8, grid_res)
xx, yy = np.meshgrid(x_lin, y_lin)
grid_pts = xx + 1j * yy
grid_flat = grid_pts.flatten()

# 1. 8-PSK Decision Regions
_, psk8_s, _, _, _ = get_mpsk_constellation(8)
dists_psk = np.abs(grid_flat[:, np.newaxis] - psk8_s[np.newaxis, :]) ** 2
classes_psk = np.argmin(dists_psk, axis=1).reshape(xx.shape)

ax1.contourf(xx, yy, classes_psk, levels=np.arange(9) - 0.5, cmap='tab10', alpha=0.35)
ax1.scatter(np.real(psk8_s), np.imag(psk8_s), color='black', s=70, zorder=5)
for m in range(8):
    angle = (2 * m) * np.pi / 8
    ax1.plot([0, 1.8 * np.cos(angle)], [0, 1.8 * np.sin(angle)], 'k--', linewidth=1.0, alpha=0.7)
ax1.set_title('8-PSK Decision Regions (Radial Pie Wedges)', fontsize=13)
ax1.set_xlabel('In-Phase (I)')
ax1.set_ylabel('Quadrature (Q)')
ax1.set_xlim(-1.8, 1.8); ax1.set_ylim(-1.8, 1.8); ax1.grid(True); ax1.set_aspect('equal')

# 2. 16-QAM Decision Regions
_, qam16_s, _, _, _ = get_mqam_constellation(16)
dists_qam = np.abs(grid_flat[:, np.newaxis] - qam16_s[np.newaxis, :]) ** 2
classes_qam = np.argmin(dists_qam, axis=1).reshape(xx.shape)

ax2.contourf(xx, yy, classes_qam, levels=np.arange(17) - 0.5, cmap='tab20', alpha=0.35)
ax2.scatter(np.real(qam16_s), np.imag(qam16_s), color='black', s=60, zorder=5)
c_norm = 1.0 / np.sqrt(10.0)
for boundary in [-2.0 * c_norm, 0.0, 2.0 * c_norm]:
    ax2.axvline(boundary, color='black', linestyle='--', linewidth=1.2, alpha=0.8)
    ax2.axhline(boundary, color='black', linestyle='--', linewidth=1.2, alpha=0.8)
ax2.set_title('16-QAM Decision Regions (Square Voronoi Cells)', fontsize=13)
ax2.set_xlabel('In-Phase (I)')
ax2.set_ylabel('Quadrature (Q)')
ax2.set_xlim(-1.8, 1.8); ax2.set_ylim(-1.8, 1.8); ax2.grid(True); ax2.set_aspect('equal')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / '3_decision_regions_and_voronoi_boundaries.png')
plt.show()
print(f'[OK] Figure 3 saved to: {OUTPUT_DIR / "3_decision_regions_and_voronoi_boundaries.png"}')

---
## 6. Gray Mapping vs Natural (Binary) Mapping Comparison [Figure 4]

We evaluate the Bit Error Rate penalty when using natural binary mapping versus Gray coding on 16-QAM:
- **Gray Mapping:** Adjacent symbols differ by only 1 bit ($00 \to 01 \to 11 \to 10$). Single symbol detection errors across nearest boundaries produce only 1 bit error ($P_b \approx P_s / 4$).
- **Natural Binary Mapping:** Transition from level $-1$ to $+1$ corresponds to $01 \to 10$, flipping **2 bits simultaneously** across the most sensitive zero boundary! Consequently, natural mapping exhibits a substantially higher Bit Error Rate despite having identical Symbol Error Rate (SER).

In [ ]:
# Cell 7: Gray vs Binary Mapping Performance Comparison on 16-QAM [Figure 4]
eb_no_db = np.linspace(0.0, 14.0, 15)

def simulate_qam_mapping(eb_range: np.ndarray, n_symbols: int = 40000):
    _, _, map_gray, _, _ = get_mqam_constellation(16, use_gray=True)
    _, _, map_bin, _, _ = get_mqam_constellation(16, use_gray=False)

    rng = np.random.default_rng(42)
    indices = rng.integers(0, 16, size=n_symbols)
    tx_gray = np.array(list(map_gray.values()))[indices]
    tx_bin = np.array(list(map_bin.values()))[indices]

    bits_gray = np.array([int(b) for idx in indices for b in list(map_gray.keys())[idx]])
    bits_bin = np.array([int(b) for idx in indices for b in list(map_bin.keys())[idx]])

    ber_gray = np.zeros(len(eb_range))
    ber_bin = np.zeros(len(eb_range))
    ser_all = np.zeros(len(eb_range))

    for s_idx, snr_db in enumerate(eb_range):
        eb = 1.0 / 4.0
        n0 = eb / (10.0 ** (snr_db / 10.0))
        sigma = np.sqrt(n0 / 2.0)
        noise = rng.normal(0.0, sigma, size=n_symbols) + 1j * rng.normal(0.0, sigma, size=n_symbols)

        rx_g = tx_gray + noise
        rx_b = tx_bin + noise

        det_g_bits, det_g_syms = demodulate_minimum_distance(rx_g, map_gray)
        det_b_bits, _ = demodulate_minimum_distance(rx_b, map_bin)

        ber_gray[s_idx] = np.mean(det_g_bits != bits_gray)
        ber_bin[s_idx] = np.mean(det_b_bits != bits_bin)
        ser_all[s_idx] = np.mean(det_g_syms != tx_gray)

    return ser_all, ber_gray, ber_bin

print('Simulating 16-QAM Gray vs Binary mapping (40,000 symbols per SNR point)...')
ser_all, ber_gray, ber_bin = simulate_qam_mapping(eb_no_db, n_symbols=40000)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# (a) 1D Slice comparison
levels = np.array([-3, -1, 1, 3]) / np.sqrt(10.0)
gray_labels = ['00', '01', '11', '10']
bin_labels = ['00', '01', '10', '11']

ax1.axhline(0, color='black', linewidth=1.5)
ax1.scatter(levels, [0]*4, color='crimson', s=150, zorder=5)
for i, x_val in enumerate(levels):
    ax1.annotate(f"Gray: '{gray_labels[i]}'\nBinary: '{bin_labels[i]}'", (x_val, 0), textcoords='offset points',
                 xytext=(0, 25 if i % 2 == 0 else -40), ha='center', fontsize=10, fontweight='bold',
                 bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='gray'))
ax1.set_title('1D Slice Bit Assignments (Levels: -3, -1, +1, +3)', fontsize=12)
ax1.set_xlabel('In-Phase Axis Coordinate')
ax1.set_xlim(-1.5, 1.5); ax1.set_ylim(-0.3, 0.3)
ax1.set_yticks([])
ax1.grid(True, linestyle='--', alpha=0.5)

# (b) Semilog BER comparison
ax2.semilogy(eb_no_db, ser_all, 'k--', linewidth=1.8, label='Symbol Error Rate SER (Identical)')
ax2.semilogy(eb_no_db, ber_gray, 'ro-', linewidth=1.8, label='BER with Gray Mapping ($P_b \\approx P_s / 4$)')
ax2.semilogy(eb_no_db, ber_bin, 'bs-', linewidth=1.8, label='BER with Binary Mapping (Higher Errors)')

ax2.set_title('16-QAM BER: Gray vs Binary Mapping over AWGN', fontsize=12)
ax2.set_xlabel('Energy per Bit to Noise Spectral Density $E_b/N_0$ (dB)', fontsize=11)
ax2.set_ylabel('Error Probability', fontsize=11)
ax2.set_xlim(0, 14); ax2.set_ylim(1e-5, 1.0)
ax2.grid(True, which='both', linestyle='--', alpha=0.6)
ax2.legend(loc='lower left', framealpha=0.95)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / '4_gray_vs_binary_mapping_comparison.png')
plt.show()
print(f'[OK] Figure 4 saved to: {OUTPUT_DIR / "4_gray_vs_binary_mapping_comparison.png"}')

---
## 7. Peak-to-Average Power Ratio (PAPR) & Concentric Energy Tiers [Figure 7]

While M-PSK maintains a constant amplitude with $\text{PAPR} = 0\text{ dB}$, M-QAM exhibits significant envelope variation:
$$\text{PAPR} = \frac{\max_m |s_m|^2}{E_{\text{avg}}}$$
- **16-QAM Concentric Energy Tiers:**
  1. **Inner Tier (4 corner points):** $|s|^2 = (1^2 + 1^2)/10 = 0.2$ (25% of symbols)
  2. **Middle Tier (8 edge points):** $|s|^2 = (1^2 + 3^2)/10 = 1.0$ (50% of symbols)
  3. **Outer Tier (4 outer corners):** $|s|^2 = (3^2 + 3^2)/10 = 1.8$ (25% of symbols)
  - $\text{PAPR}_{16\text{-QAM}} = 1.8 / 1.0 = 1.8 \implies \mathbf{2.55\text{ dB}}$.
- **64-QAM:** $\text{PAPR} = (7^2 + 7^2)/42 = 98/42 = 2.333 \implies \mathbf{3.68\text{ dB}}$.
- High PAPR requires higher power amplifier back-off to operate in the linear regime and prevent distortion.

In [ ]:
# Cell 8: Peak-to-Average Power Ratio (PAPR) & Concentric Energy Tiers [Figure 7]
_, qpsk_s, _, _, _ = get_mpsk_constellation(4)
_, psk8_s, _, _, _ = get_mpsk_constellation(8)
_, qam16_s, _, _, _ = get_mqam_constellation(16)
_, qam64_s, _, _, _ = get_mqam_constellation(64)

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# (a) PAPR Bar Chart
schemes = ['QPSK', '8-PSK', '16-QAM', '64-QAM']
papr_vals = [
    np.max(np.abs(qpsk_s)**2) / np.mean(np.abs(qpsk_s)**2),
    np.max(np.abs(psk8_s)**2) / np.mean(np.abs(psk8_s)**2),
    np.max(np.abs(qam16_s)**2) / np.mean(np.abs(qam16_s)**2),
    np.max(np.abs(qam64_s)**2) / np.mean(np.abs(qam64_s)**2)
]
papr_db = [10.0 * np.log10(v) for v in papr_vals]

bars = axes[0].bar(schemes, papr_db, color=['royalblue', 'forestgreen', 'crimson', 'purple'], edgecolor='black', width=0.55)
for bar in bars:
    h = bar.get_height()
    axes[0].text(bar.get_x() + bar.get_width()/2.0, h + 0.1, f'{h:.2f} dB', ha='center', va='bottom', fontweight='bold')
axes[0].set_ylabel('PAPR (dB)')
axes[0].set_ylim(0, 4.5)
axes[0].set_title('Peak-to-Average Power Ratio (PAPR) of Constellations', fontsize=12)
axes[0].grid(True, axis='y', linestyle='--', alpha=0.6)

# (b) 16-QAM Concentric Energy Tiers
p_qam16 = np.abs(qam16_s)**2
unique_p, counts = np.unique(np.round(p_qam16, 4), return_counts=True)

axes[1].bar([f'Ring {i+1}\n($|s|^2={p:.1f}$)' for i, p in enumerate(unique_p)], counts,
            color='darkorange', edgecolor='black', width=0.45)
for i, c in enumerate(counts):
    axes[1].text(i, c + 0.2, f'{c} symbols\n({c/16*100:.0f}%)', ha='center', va='bottom', fontweight='bold')
axes[1].set_ylabel('Number of Symbols')
axes[1].set_ylim(0, 10)
axes[1].set_title('16-QAM Distinct Symbol Energy Tiers (3 Concentric Rings)', fontsize=12)
axes[1].grid(True, axis='y', linestyle='--', alpha=0.6)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / '7_constellation_power_and_peak_to_average_ratio.png')
plt.show()
print(f'[OK] Figure 7 saved to: {OUTPUT_DIR / "7_constellation_power_and_peak_to_average_ratio.png"}')

---
## 8. Spectral Efficiency vs Required Eb/N0 & Shannon Capacity Limit [Figure 6]

As spectral efficiency $\eta = \log_2(M)$ increases, higher $E_b/N_0$ is required to maintain target error probability $\text{BER} = 10^{-3}$:
- **Shannon Channel Capacity Bound:** $C/B = \log_2(1 + S/N) = \log_2(1 + \eta \cdot E_b/N_0) \implies \left(\frac{E_b}{N_0}\right)_{\min} = \frac{2^\eta - 1}{\eta}$.
- **16-QAM vs 16-PSK:** For the same spectral efficiency $\eta = 4\text{ bits/s/Hz}$, 16-QAM requires $10.5\text{ dB}$, whereas 16-PSK requires $14.8\text{ dB}$. This demonstrates a massive **$\approx 4.3\text{ dB}$ power advantage** in favor of 16-QAM due to Cartesian 2D lattice packing!

In [ ]:
# Cell 9: Spectral Efficiency vs Power Trade-Off & Shannon Bound [Figure 6]
schemes = [
    ('QPSK', 2, 6.8, 'royalblue', 'o'),
    ('8-PSK', 3, 10.3, 'forestgreen', '^'),
    ('16-QAM', 4, 10.5, 'crimson', 's'),
    ('16-PSK', 4, 14.8, 'orange', 'D'),
    ('64-QAM', 6, 14.5, 'purple', 'p')
]

fig, ax = plt.subplots(figsize=(10.5, 6.5))

eta_shannon = np.linspace(0.5, 7.0, 200)
eb_no_shannon_lin = (2.0 ** eta_shannon - 1.0) / eta_shannon
eb_no_shannon_db = 10.0 * np.log10(eb_no_shannon_lin)

ax.plot(eta_shannon, eb_no_shannon_db, 'k--', linewidth=2.0, label='Shannon Capacity Limit ($C/B = \\log_2(1 + \\text{SNR})$)')
ax.axhline(-1.59, color='gray', linestyle=':', label='Ultimate Shannon Limit (-1.59 dB)')

for name, eta, req_ebno, col, marker in schemes:
    ax.scatter(eta, req_ebno, color=col, marker=marker, s=120, edgecolors='black', linewidth=1.5, zorder=5)
    ax.annotate(f'{name}\n({req_ebno:.1f} dB)', (eta, req_ebno), textcoords='offset points',
                xytext=(15, -5 if '16-PSK' in name else 10),
                fontsize=10, fontweight='bold', color=col,
                bbox=dict(boxstyle='round,pad=0.2', fc='white', ec=col, lw=1.2))

# Highlight 16-QAM vs 16-PSK gap
ax.annotate('$\\approx 4.3$ dB Power Savings\nof 16-QAM over 16-PSK!',
            xy=(4, 10.5), xytext=(4.3, 12.3),
            arrowprops=dict(arrowstyle='->', color='darkred', lw=1.5),
            fontsize=10, fontweight='bold', color='darkred')

ax.set_title('Bandwidth Efficiency ($\\eta$) vs Required $E_b/N_0$ at $\\text{BER} = 10^{-3}$', fontsize=13)
ax.set_xlabel('Spectral Efficiency $\\eta = \\log_2(M)$ (bits/s/Hz)', fontsize=12)
ax.set_ylabel('Required $E_b/N_0$ (dB) for $\\text{BER} = 10^{-3}$', fontsize=12)
ax.set_xlim(1.0, 7.0)
ax.set_ylim(-3.0, 20.0)
ax.grid(True, linestyle='--', alpha=0.6)
ax.legend(loc='upper left', framealpha=0.95)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / '6_ber_vs_spectral_efficiency.png')
plt.show()
print(f'[OK] Figure 6 saved to: {OUTPUT_DIR / "6_ber_vs_spectral_efficiency.png"}')

---
## 9. Full Monte Carlo BER Performance vs Theoretical Bounds [Figure 5]

We perform full Monte Carlo simulations for QPSK, 8-PSK, 16-QAM, and 64-QAM over $E_b/N_0 \in [0, 16]\text{ dB}$, and validate against analytical closed-form equations:
- **M-PSK Theory:** $P_s \approx 2 Q\left(\sqrt{2k E_b/N_0}\sin(\pi/M)\right), \quad P_b \approx P_s / k$.
- **Square M-QAM Theory:** $P_{\sqrt{M}} = 2\left(1 - \frac{1}{\sqrt{M}}\right) Q\left(\sqrt{\frac{3k E_b}{(M-1)N_0}}\right), \quad P_s = 1 - (1 - P_{\sqrt{M}})^2, \quad P_b \approx P_s / k$.

In [ ]:
# Cell 10: Monte Carlo BER Simulations vs Digital Communication Theory [Figure 5]
def simulate_ser_ber(m_order: int, mod_type: str, eb_no_db_range: np.ndarray,
                     use_gray: bool = True, n_symbols: int = 25000) -> Tuple[np.ndarray, np.ndarray]:
    k = int(np.log2(m_order))
    if mod_type == 'PSK':
        _, _, mapping, _, _ = get_mpsk_constellation(m_order)
    else:
        _, _, mapping, _, _ = get_mqam_constellation(m_order, use_gray=use_gray)

    bit_strings = list(mapping.keys())
    constellation = np.array(list(mapping.values()))

    rng = np.random.default_rng(42)
    sym_indices = rng.integers(0, m_order, size=n_symbols)
    tx_symbols = constellation[sym_indices]

    tx_bits = np.empty(n_symbols * k, dtype=int)
    for i, idx in enumerate(sym_indices):
        b_str = bit_strings[idx]
        for b_pos in range(k):
            tx_bits[i * k + b_pos] = int(b_str[b_pos])

    ser_arr = np.zeros(len(eb_no_db_range))
    ber_arr = np.zeros(len(eb_no_db_range))

    for s_idx, snr_db in enumerate(eb_no_db_range):
        rx = add_awgn_complex(tx_symbols, snr_db, bits_per_symbol=k, seed=1000 + s_idx)
        rx_bits, rx_syms = demodulate_minimum_distance(rx, mapping)
        ser_arr[s_idx] = np.mean(rx_syms != tx_symbols)
        ber_arr[s_idx] = np.mean(rx_bits != tx_bits)

    return ser_arr, ber_arr

def theoretical_ser_ber(m_order: int, mod_type: str, eb_no_db_range: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    eb_no_lin = 10.0 ** (eb_no_db_range / 10.0)
    k = int(np.log2(m_order))
    def q_func(x): return 0.5 * sp.erfc(x / np.sqrt(2.0))

    if mod_type == 'PSK':
        arg = np.sqrt(2.0 * k * eb_no_lin) * np.sin(np.pi / m_order)
        ser = 2.0 * q_func(arg)
        ber = ser / k
    else:
        sqrt_m = int(np.sqrt(m_order))
        arg = np.sqrt((3.0 * k * eb_no_lin) / (m_order - 1.0))
        p_sub = 2.0 * (1.0 - 1.0 / sqrt_m) * q_func(arg)
        ser = 1.0 - (1.0 - p_sub) ** 2
        ber = ser / k
    return ser, ber

eb_no_db = np.linspace(0.0, 16.0, 9)
eb_fine = np.linspace(0.0, 16.0, 100)

print('Running Monte Carlo simulations for QPSK, 8-PSK, 16-QAM, and 64-QAM...')
_, ber_qpsk = simulate_ser_ber(4, 'PSK', eb_no_db, n_symbols=25000)
_, ber_psk8 = simulate_ser_ber(8, 'PSK', eb_no_db, n_symbols=25000)
_, ber_qam16 = simulate_ser_ber(16, 'QAM', eb_no_db, n_symbols=25000)
_, ber_qam64 = simulate_ser_ber(64, 'QAM', eb_no_db, n_symbols=25000)

_, th_qpsk = theoretical_ser_ber(4, 'PSK', eb_fine)
_, th_psk8 = theoretical_ser_ber(8, 'PSK', eb_fine)
_, th_qam16 = theoretical_ser_ber(16, 'QAM', eb_fine)
_, th_qam64 = theoretical_ser_ber(64, 'QAM', eb_fine)

fig, ax = plt.subplots(figsize=(11, 7.5))

# QPSK
ax.semilogy(eb_fine, th_qpsk, 'b-', linewidth=1.8, label='QPSK (Theory)')
ax.semilogy(eb_no_db, ber_qpsk, 'bo', markersize=6, fillstyle='none', markeredgewidth=1.8, label='QPSK (Sim)')

# 8-PSK
ax.semilogy(eb_fine, th_psk8, 'g-', linewidth=1.8, label='8-PSK (Theory)')
ax.semilogy(eb_no_db, ber_psk8, 'g^', markersize=6, fillstyle='none', markeredgewidth=1.8, label='8-PSK (Sim)')

# 16-QAM
ax.semilogy(eb_fine, th_qam16, 'r-', linewidth=1.8, label='16-QAM (Theory)')
ax.semilogy(eb_no_db, ber_qam16, 'rs', markersize=6, fillstyle='none', markeredgewidth=1.8, label='16-QAM (Sim)')

# 64-QAM
ax.semilogy(eb_fine, th_qam64, 'm-', linewidth=1.8, label='64-QAM (Theory)')
ax.semilogy(eb_no_db, ber_qam64, 'md', markersize=6, fillstyle='none', markeredgewidth=1.8, label='64-QAM (Sim)')

ax.set_title('Bit Error Rate (BER) Comparison for M-PSK and M-QAM over AWGN', fontsize=13)
ax.set_xlabel('Energy per Bit to Noise Spectral Density $E_b/N_0$ (dB)', fontsize=12)
ax.set_ylabel('Bit Error Rate (BER)', fontsize=12)
ax.set_xlim(0, 16); ax.set_ylim(1e-5, 0.6)
ax.grid(True, which='both', linestyle='--', alpha=0.6)
ax.legend(loc='lower left', framealpha=0.95)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / '5_ber_ser_curves_mpsk_mqam.png')
plt.show()
print(f'[OK] Figure 5 saved to: {OUTPUT_DIR / "5_ber_ser_curves_mpsk_mqam.png"}')

---
## 10. Summary Findings & Engineering Conclusions

1. **Constellation Energy Normalization:** Enforcing $E_s = 1.0$ through scale factor $c = 1/\sqrt{E_{\text{unnorm}}}$ ensures that comparative benchmarking against $E_b/N_0$ is mathematically rigorous and fair across all modulation orders.
2. **QAM Power Superiority over High-Order PSK:** 16-QAM saves $\approx 4.3\text{ dB}$ in transmit power compared to 16-PSK for the same $4\text{ bits/s/Hz}$ bandwidth efficiency, because 2D Cartesian lattice packing preserves larger minimum Euclidean distance than 1D circular phase packing.
3. **Gray Coding Efficiency:** 2D Gray mapping ensures that single boundary crossings flip only 1 bit, reducing BER by a factor of 2 to 3 compared to natural binary mapping ($P_b \approx P_s / \log_2(M)$).
4. **Separable Slicing Complexity:** Square M-QAM decision regions are rectangular, allowing the 2D Euclidean search to decouple into two 1D slicings on $I$ and $Q$, drastically cutting receiver complexity from $O(M)$ to $O(2\sqrt{M})$.
5. **PAPR Trade-off:** High-order QAM incurs a PAPR penalty ($2.55\text{ dB}$ for 16-QAM, $3.68\text{ dB}$ for 64-QAM), requiring power amplifier back-off to operate linearly.